# TEM/STEM nanoparticle polydispersity — Thermo Fisher `.emd` files

Dedicated pipeline for **Thermo Fisher Velox `.emd`** files. Pixel calibration and acquisition mode (bright-field vs dark-field/HAADF) are read directly from the file's own metadata via HyperSpy — you only need to point it at a file (or a folder of files).

**Pipeline**: load `.emd` + metadata → normalize to 8-bit → Otsu threshold (direction chosen automatically from bright/dark-field) → watershed split of touching particles → measure diameters → polydispersity statistics + histogram + overlay.

**Dependencies**: `hyperspy`, `opencv-python-headless`, `numpy`, `scipy`, `scikit-image`, `matplotlib`, `pandas` (see `requirements.txt`).

**If no particles are found**: the pipeline does not crash — it shows you the segmentation mask instead, so you can see why (wrong bright/dark-field guess, or filters too strict) and adjust the parameters in Configuration.

## Configuration

`file_path` is used by the **Single file** section, `folder_path` by **Batch processing**. The rest have sensible defaults — only touch them if particle detection doesn't work well (see the notes on each parameter).

In [ ]:
file_path = "./Data/Velox/STEM HAADF 1532.emd"  # <-- path to a single .emd file (used by "Single file" section)
folder_path = "./Data/Velox"                     # <-- folder containing .emd files (used by "Batch processing" section)

min_diam_px = 25          # minimum accepted particle diameter, in pixels (filters out noise)
min_circularity = 0.5     # minimum accepted circularity (1.0 = perfect circle)
min_peak_distance = 15    # minimum distance between particle centers for watershed, in px (lower for small/closely packed particles)
bright_particles_override = None  # None = auto-detect bright/dark-field from metadata; set True or False to force it if the guess is wrong
outdir = "Result_Poly_emd"  # set to None to skip saving overlay.png/histogram.png to disk

In [ ]:
import os
import glob

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import matplotlib.pyplot as plt
import hyperspy.api as hs

%matplotlib inline

## 1. Loading `.emd` files and metadata

`load_emd_image()` reads the pixel calibration (nm/px) and detector name from the file's metadata, normalizes the (often 16-bit-range) raw detector counts to 8-bit, and guesses bright-field vs dark-field from the detector name (e.g. `HAADF`, `DF-S` → dark-field; `BF-S` → bright-field). Override with `bright_particles_override` in Configuration if it guesses wrong for your detector naming.

In [ ]:
UNIT_TO_NM = {
    "nm": 1.0,
    "um": 1000.0,
    "\u00b5m": 1000.0,
    "pm": 0.001,
    "a": 0.1,
    "\u00e5": 0.1,
    "angstrom": 0.1,
    "m": 1e9,
}


def normalize_to_uint8(data, low_percentile=1.0, high_percentile=99.0):
    """
    Rescales an arbitrary-range array (e.g. raw STEM detector counts) to
    0-255 uint8, clipping the extreme percentiles first so a few outlier
    pixels don't compress the contrast of the rest of the image.
    """
    lo, hi = np.percentile(data, [low_percentile, high_percentile])
    if hi <= lo:
        lo, hi = float(data.min()), float(data.max())
    clipped = np.clip(data, lo, hi)
    scaled = (clipped - lo) / (hi - lo + 1e-12) * 255.0
    return scaled.astype(np.uint8)


def classify_darkfield(detector_name):
    """
    Guesses bright-field vs dark-field from a Thermo Fisher detector name
    (e.g. 'HAADF', 'DF-S' -> dark-field; 'BF-S' -> bright-field). Defaults
    to bright-field (False) if the name is missing or unrecognized.
    """
    if not detector_name:
        return False
    return "DF" in str(detector_name).upper()


def load_emd_image(path, signal_index=0):
    """
    Loads a Thermo Fisher Velox .emd file with HyperSpy and returns
    (gray_uint8, nm_per_px, is_darkfield, info). Calibration and
    acquisition mode are read from the file's metadata -- no scale bar
    detection needed. If the file contains multiple signals, the one at
    signal_index is used (default: the first).
    """
    result = hs.load(path)
    signals = result if isinstance(result, list) else [result]
    if len(signals) > 1:
        print("Note: " + str(len(signals)) + " signals found in '" + path +
              "', using index " + str(signal_index) +
              " (" + str(signals[signal_index].metadata.General.title) + ")")
    s = signals[signal_index]

    ax = s.axes_manager.signal_axes[0]
    unit_factor = UNIT_TO_NM.get(str(ax.units).lower(), 1.0)
    nm_per_px = float(ax.scale) * unit_factor

    om = s.original_metadata.as_dictionary()
    detector_name = om.get("BinaryResult", {}).get("Detector") or s.metadata.General.title
    is_darkfield = classify_darkfield(detector_name)

    gray = normalize_to_uint8(s.data)

    info = {
        "detector": detector_name,
        "raw_dtype": str(s.data.dtype),
        "raw_min": float(s.data.min()),
        "raw_max": float(s.data.max()),
    }
    return gray, nm_per_px, is_darkfield, info

## 2. Segmentation (Otsu + watershed)

In [ ]:
def segment_and_split_particles(gray_img, min_peak_distance=15, bright_particles=False):
    """
    Segments particles on a uniform background and splits touching
    particles using watershed based on the distance transform.

    Parameters
    ----------
    gray_img : np.ndarray (2D, uint8)
    min_peak_distance : int
        Minimum distance (in pixels) between local maxima of the distance
        transform, used as markers for the watershed. Should be chosen
        based on the approximate particle radius (smaller if particles are
        small/closely packed).
    bright_particles : bool
        False for bright-field-style images where particles are darker
        than the background. True for dark-field/HAADF STEM images, where
        particles are the brightest pixels (Z-contrast).

    Returns
    -------
    labels_ws : np.ndarray (2D, int) - watershed labels (0 = background)
    mask : np.ndarray (2D, bool) - binary mask of the particles
    """
    blur = cv2.GaussianBlur(gray_img, (5, 5), 0)
    thresh_type = cv2.THRESH_BINARY if bright_particles else cv2.THRESH_BINARY_INV
    _, th = cv2.threshold(blur, 0, 255, thresh_type + cv2.THRESH_OTSU)

    kernel = np.ones((3, 3), np.uint8)
    th_clean = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)
    th_clean = cv2.morphologyEx(th_clean, cv2.MORPH_CLOSE, kernel, iterations=1)
    mask = th_clean > 0

    dist = ndi.distance_transform_edt(mask)
    coords = peak_local_max(dist, min_distance=min_peak_distance, labels=mask,
                             exclude_border=False)
    markers_mask = np.zeros(dist.shape, dtype=bool)
    markers_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(markers_mask)

    labels_ws = watershed(-dist, markers, mask=mask)
    return labels_ws, mask

## 3. Particle measurement

In [ ]:
def measure_particles(labels_ws, nm_per_px, min_diam_px=25, min_circularity=0.5):
    """
    Computes area, equivalent diameter and circularity for each watershed
    region and filters out regions that are too small or not circular
    enough (typically noise fragments or edge artifacts).

    Returns
    -------
    list of dict, one per accepted particle, with keys:
        label, area_px, eq_diam_px, eq_diam_nm, circularity, solidity, centroid
    """
    props = regionprops(labels_ws)
    particles = []
    for p in props:
        area = p.area
        perim = p.perimeter if p.perimeter > 0 else 1
        circularity = 4 * np.pi * area / (perim ** 2)
        eq_diam_px = 2 * np.sqrt(area / np.pi)

        if eq_diam_px < min_diam_px:
            continue
        if circularity < min_circularity:
            continue

        particles.append({
            "label": p.label,
            "area_px": area,
            "eq_diam_px": eq_diam_px,
            "eq_diam_nm": eq_diam_px * nm_per_px,
            "circularity": circularity,
            "solidity": p.solidity,
            "centroid": p.centroid,  # (row, col)
        })
    return particles

## 4. Polydispersity statistics

In [ ]:
def polydispersity_stats(diam_nm):
    """
    Computes polydispersity statistics from a non-empty array of
    diameters (in nm). Only ever called with at least one particle --
    analyze_emd_image() handles the zero-particle case separately.

    PDI here is defined as (sigma/mean)^2, the same convention typically
    used in DLS. It is a dimensionless metric: lower values indicate a
    more monodisperse sample (e.g. PDI < 0.05 - 0.1 in TEM).
    """
    diam_nm = np.asarray(diam_nm)
    mean_d = diam_nm.mean()
    std_d = diam_nm.std(ddof=1) if diam_nm.size > 1 else 0.0
    return {
        "n": len(diam_nm),
        "mean_nm": mean_d,
        "median_nm": float(np.median(diam_nm)),
        "std_nm": std_d,
        "cv_percent": std_d / mean_d * 100,
        "pdi": (std_d / mean_d) ** 2,
        "min_nm": float(diam_nm.min()),
        "max_nm": float(diam_nm.max()),
    }

## 5. Visualization

In [ ]:
def draw_overlay(color_img, particles, out_path=None):
    overlay = color_img.copy()
    for p in particles:
        cy, cx = p["centroid"]
        r = p["eq_diam_px"] / 2
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), int(round(r)),
                   (0, 0, 255), 2)
        cv2.circle(overlay, (int(round(cx)), int(round(cy))), 1, (0, 255, 0), 2)
    if out_path:
        cv2.imwrite(out_path, overlay)
    return overlay


def show_overlay(overlay_bgr, n_particles=None):
    """Displays a BGR (OpenCV) overlay image inline using matplotlib."""
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB))
    title = "Detected particles"
    if n_particles is not None:
        title += f" (N={n_particles})"
    plt.title(title)
    plt.axis("off")
    plt.show()


def plot_histogram(diam_nm, stats, out_path=None, material="nanoparticles", show=True):
    plt.figure(figsize=(7, 5))
    plt.hist(diam_nm, bins=30, color="#4C72B0", edgecolor="white")
    plt.axvline(stats["mean_nm"], color="crimson", linestyle="--", linewidth=1.5,
                label=f'Mean = {stats["mean_nm"]:.2f} nm')
    plt.xlabel("Particle diameter (nm)")
    plt.ylabel("Count")
    plt.title(
        f'Size distribution {material} (N={stats["n"]})\n'
        f'Std dev = {stats["std_nm"]:.2f} nm, CV = {stats["cv_percent"]:.1f}%, '
        f'PDI = {stats["pdi"]:.4f}'
    )
    plt.legend()
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close()

## Full pipeline (single file)

`analyze_emd_image()` is purely computational (loads, segments, measures, computes stats) and never crashes on zero detected particles -- `stats`/`overlay` are simply `None` in that case. `show_results()` and `save_results()` handle displaying/saving, including showing the segmentation mask when nothing was detected, so you can see why.

In [ ]:
def analyze_emd_image(path, min_diam_px=25, min_circularity=0.5, min_peak_distance=15,
                       bright_particles_override=None, emd_signal_index=0):
    """
    Runs the full pipeline on a single .emd file (load + metadata
    calibration, segmentation, measurement, statistics) and returns a
    dict: path, nm_per_px, is_darkfield, detector, gray, mask, particles,
    stats, overlay. If no particles are detected, stats and overlay are
    None -- use show_results() to see the segmentation mask and debug why
    (wrong bright/dark-field guess, or min_diam_px/min_circularity too
    strict) instead of crashing.
    """
    gray, nm_per_px, is_darkfield, info = load_emd_image(path, signal_index=emd_signal_index)
    if bright_particles_override is not None:
        is_darkfield = bright_particles_override

    labels_ws, mask = segment_and_split_particles(
        gray, min_peak_distance=min_peak_distance, bright_particles=is_darkfield,
    )
    particles = measure_particles(
        labels_ws, nm_per_px=nm_per_px,
        min_diam_px=min_diam_px, min_circularity=min_circularity,
    )

    stats = None
    overlay = None
    if particles:
        diam_nm = np.array([p["eq_diam_nm"] for p in particles])
        stats = polydispersity_stats(diam_nm)
        color_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
        overlay = draw_overlay(color_img, particles)

    return {
        "path": path,
        "nm_per_px": nm_per_px,
        "is_darkfield": is_darkfield,
        "detector": info["detector"],
        "gray": gray,
        "mask": mask,
        "particles": particles,
        "stats": stats,
        "overlay": overlay,
    }


def show_results(result, material="nanoparticles"):
    """
    Displays the metadata summary and, if particles were found, the
    overlay + size histogram. If none were found, shows the segmentation
    mask instead so you can see why.
    """
    print("File: " + result["path"])
    print("Detector: " + str(result["detector"]) + " (dark-field: " + str(result["is_darkfield"]) + ")")
    print("Calibration: " + format(result["nm_per_px"], ".4f") + " nm/px")

    if not result["particles"]:
        print("No particles detected -- showing the segmentation mask for debugging. "
              "Try bright_particles_override, or relax min_diam_px / min_circularity.")
        plt.figure(figsize=(8, 8))
        plt.imshow(result["mask"], cmap="gray")
        plt.title("Segmentation mask (no particles passed the filters)")
        plt.axis("off")
        plt.show()
        return

    show_overlay(result["overlay"], n_particles=result["stats"]["n"])
    diam_nm = np.array([p["eq_diam_nm"] for p in result["particles"]])
    plot_histogram(diam_nm, result["stats"], material=material)


def save_results(result, outdir, material="nanoparticles"):
    """Saves overlay.png and histogram.png for one result into outdir (skipped if no particles were found)."""
    if not result["particles"]:
        print("Skipping save for '" + result["path"] + "': no particles detected.")
        return
    os.makedirs(outdir, exist_ok=True)
    cv2.imwrite(os.path.join(outdir, "overlay.png"), result["overlay"])
    diam_nm = np.array([p["eq_diam_nm"] for p in result["particles"]])
    plot_histogram(diam_nm, result["stats"], os.path.join(outdir, "histogram.png"),
                    material=material, show=False)

## Run: single file

Uses `file_path` from Configuration.

In [ ]:
result = analyze_emd_image(
    file_path,
    min_diam_px=min_diam_px, min_circularity=min_circularity, min_peak_distance=min_peak_distance,
    bright_particles_override=bright_particles_override,
)

show_results(result)

if outdir:
    save_results(result, outdir)

result["stats"]

## Batch processing (multiple `.emd` files)

Uses `folder_path` from Configuration. Every `.emd` file directly inside that folder is analyzed; files with 0 detected particles are kept in the results (not skipped) so you can inspect them, files that error out unexpectedly (e.g. unreadable metadata) are skipped with a printed warning.

In [ ]:
def analyze_emd_folder(folder_path, min_diam_px=25, min_circularity=0.5, min_peak_distance=15,
                        bright_particles_override=None, emd_signal_index=0):
    """
    Runs analyze_emd_image() on every .emd file directly inside
    folder_path (non-recursive) and returns a dict {file_name: result}.
    """
    emd_paths = sorted(glob.glob(os.path.join(folder_path, "*.emd")))
    if not emd_paths:
        raise FileNotFoundError("No .emd files found in: " + folder_path)

    results = {}
    for path in emd_paths:
        name = os.path.splitext(os.path.basename(path))[0]
        try:
            res = analyze_emd_image(
                path, min_diam_px=min_diam_px, min_circularity=min_circularity,
                min_peak_distance=min_peak_distance,
                bright_particles_override=bright_particles_override,
                emd_signal_index=emd_signal_index,
            )
        except Exception as exc:
            print("[SKIPPED] " + name + ": " + str(exc))
            continue

        results[name] = res
        if res["stats"] is None:
            print(name + ": 0 particles detected (detector: " + str(res["detector"]) +
                  ", dark-field: " + str(res["is_darkfield"]) + ")")
        else:
            s = res["stats"]
            print(name + ": N=" + str(s["n"]) + ", mean=" + format(s["mean_nm"], ".2f") +
                  " nm, CV=" + format(s["cv_percent"], ".1f") + "%, PDI=" + format(s["pdi"], ".4f"))

    return results


def summarize_folder_results(results):
    """Builds a one-row-per-file summary table (rows with 0 particles show n=0 and NaN elsewhere)."""
    rows = []
    for name, res in results.items():
        row = {"image": name, "detector": res["detector"], "is_darkfield": res["is_darkfield"]}
        if res["stats"] is not None:
            row.update(res["stats"])
        else:
            row["n"] = 0
        rows.append(row)
    return pd.DataFrame(rows).set_index("image").sort_index()

### Run batch analysis

In [ ]:
folder_results = analyze_emd_folder(
    folder_path,
    min_diam_px=min_diam_px, min_circularity=min_circularity, min_peak_distance=min_peak_distance,
    bright_particles_override=bright_particles_override,
)

for name, res in folder_results.items():
    show_results(res)
    if outdir:
        save_results(res, os.path.join(outdir, name))

summary = summarize_folder_results(folder_results)
summary

### Combined size distribution (all files pooled)

In [ ]:
all_diam_nm = np.concatenate([
    np.array([p["eq_diam_nm"] for p in res["particles"]])
    for res in folder_results.values() if res["particles"]
]) if any(res["particles"] for res in folder_results.values()) else np.array([])

if len(all_diam_nm) > 0:
    combined_stats = polydispersity_stats(all_diam_nm)
    plot_histogram(
        all_diam_nm, combined_stats,
        out_path=os.path.join(outdir, "combined_histogram.png") if outdir else None,
    )
    combined_stats
else:
    print("No particles detected in any file -- nothing to combine.")